# NLP LAB 08 — KHẢO SÁT CƠ CHẾ TOKENIZATION VÀ ATTENTION MASK CỦA BERT
**Mục tiêu Notebook:**
Khảo sát toàn diện cơ chế tiền xử lý token của mô hình Pretrained BERT:
1. Nạp Pretrained Tokenizer (`AutoTokenizer`).
2. Mã hóa các câu đánh giá thực tế từ tập dữ liệu IMDb.
3. Phân rã và trực quan hóa chuỗi Tokens, `input_ids`, `attention_mask`.
4. Tìm hiểu vai trò các Special Tokens: `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`, `[MASK]`.
5. So sánh định lượng: **Words vs Tokens vs Subword Tokens (WordPiece)**.
6. Thử nghiệm các giá trị `max_length` khác nhau.
7. Đánh giá cơ chế **Padding** và **Truncation** trong việc chuẩn hóa độ dài batch.

## 1. Thiết lập môi trường và nạp thư viện
Chúng ta sử dụng thư viện `transformers` từ Hugging Face kết hợp với `torch` và `pandas`.

In [1]:
import os
import sys
import torch
import pandas as pd
from transformers import AutoTokenizer

print("PyTorch version:", torch.__version__)
print("Python version:", sys.version.split()[0])

PyTorch version: 2.14.1+cpu
Python version: 3.13.14


## 2. Nạp Tokenizer từ Hugging Face
Mô hình Pretrained Transformer được sử dụng trong bài lab là `google/bert_uncased_L-2_H-128_A-2` (BERT-Tiny, kế thừa trọn vẹn từ điển và tokenizer WordPiece chuẩn của BERT uncased 30,522 tokens).

In [2]:
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size:,} tokens")
print(f"Model max length: {tokenizer.model_max_length}")

Tokenizer type: BertTokenizer
Vocab size: 30,522 tokens
Model max length: 1000000000000000019884624838656


## 3. Khảo sát các Special Tokens (Mục 6 W8.pdf)
BERT sử dụng 5 special tokens cơ bản đóng vai trò cốt lõi trong kiến trúc:
- `[CLS]` (Classification Token): Được chèn vào đầu mọi chuỗi đầu vào. Vectơ ẩn cuối cùng tại vị trí này đại diện cho toàn bộ chuỗi để đưa vào Classification Head.
- `[SEP]` (Separator Token): Được chèn vào cuối mỗi câu hoặc ngăn cách giữa hai phân đoạn (Segment A & Segment B).
- `[PAD]` (Padding Token): Dùng để đệm các câu ngắn về cùng độ dài cố định trong batch.
- `[UNK]` (Unknown Token): Thay thế cho các ký tự hoặc từ hoàn toàn nằm ngoài từ vựng.
- `[MASK]` (Mask Token): Dùng trong quá trình huấn luyện tự giám sát Masked Language Modeling (MLM).

In [3]:
special_tokens_info = {
    "Token Name": ["[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]"],
    "Token ID": [
        tokenizer.cls_token_id,
        tokenizer.sep_token_id,
        tokenizer.pad_token_id,
        tokenizer.unk_token_id,
        tokenizer.mask_token_id
    ],
    "Vai trò kỹ thuật": [
        "Đại diện toàn bộ chuỗi cho bài toán Sequence Classification",
        "Phân tách ranh giới câu hoặc kết thúc văn bản",
        "Token đệm để đồng bộ kích thước tensor trong mini-batch",
        "Đại diện cho từ/ký tự không thể phân tích ngoài từ điển",
        "Mặt nạ che giấu từ phục vụ hàm mất mát Masked LM"
    ]
}
pd.DataFrame(special_tokens_info)

,Token Name,Token ID,Vai trò kỹ thuật
0,[CLS],101,Đại diện toàn bộ chuỗi cho bài toán Sequence C...
1,[SEP],102,Phân tách ranh giới câu hoặc kết thúc văn bản
2,[PAD],0,Token đệm để đồng bộ kích thước tensor trong m...
3,[UNK],100,Đại diện cho từ/ký tự không thể phân tích ngoà...
4,[MASK],103,Mặt nạ che giấu từ phục vụ hàm mất mát Masked LM


## 4. Tokenize câu bài tập tính toán: `"The cats are playing."` (Mục 10 W8.pdf)
Chúng ta tiến hành mã hóa câu mẫu trong tài liệu W8.pdf để lấy kết quả đối chiếu chính xác cho file `calculations.md`.

In [4]:
text_sample = "The cats are playing."

# Mã hóa với return_tensors='pt'
encoded = tokenizer(text_sample, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
input_ids = encoded["input_ids"][0].tolist()
attention_mask = encoded["attention_mask"][0].tolist()

print("--- KẾT QUẢ MÃ HÓA CÂU MẪU ---")
print("Input Text:     ", text_sample)
print("Tokens:         ", tokens)
print("Input IDs:      ", input_ids)
print("Attention Mask: ", attention_mask)
print(f"Tổng số tokens (kèm special tokens): {len(tokens)}")

--- KẾT QUẢ MÃ HÓA CÂU MẪU ---
Input Text:      The cats are playing.
Tokens:          ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Input IDs:       [101, 1996, 8870, 2024, 2652, 1012, 102]
Attention Mask:  [1, 1, 1, 1, 1, 1, 1]
Tổng số tokens (kèm special tokens): 7


## 5. So sánh định lượng: Words vs Tokens vs Subwords
WordPiece giải quyết bài toán Out-Of-Vocabulary (OOV) bằng cách tách từ hiếm/từ phức thành các subword tokens mang tiền tố `##` (chỉ ra rằng subword này gắn liền với token trước đó mà không có khoảng trắng).

In [5]:
test_sentences = [
    "The cats are playing.",
    "Unbelievable cinematography and extraordinary direction!",
    "Transformers revolutionized Natural Language Processing completely."
]

comparison_rows = []
for s in test_sentences:
    words = s.split()
    toks_with_special = tokenizer.convert_ids_to_tokens(tokenizer.encode(s))
    toks_without_special = [t for t in toks_with_special if t not in ["[CLS]", "[SEP]"]]
    subwords = [t for t in toks_without_special if t.startswith("##")]
    
    comparison_rows.append({
        "Câu văn bản": s,
        "Số từ (Words)": len(words),
        "Số Tokens (BERT)": len(toks_without_special),
        "Số Subwords (##)": len(subwords),
        "Subwords chi tiết": ", ".join(subwords) if subwords else "None",
        "Tokens chi tiết": " | ".join(toks_without_special)
    })

pd.DataFrame(comparison_rows)

,Câu văn bản,Số từ (Words),Số Tokens (BERT),Số Subwords (##),Subwords chi tiết,Tokens chi tiết
0,The cats are playing.,4,5,0,None,the | cats | are | playing | .
1,Unbelievable cinematography and extraordinary ...,5,6,0,None,unbelievable | cinematography | and | extraord...
2,Transformers revolutionized Natural Language P...,6,8,1,##ized,transformers | revolution | ##ized | natural |...


## 6. Tokenize câu đánh giá từ tập dữ liệu IMDb thực tế
Nạp một đoạn review thực tế từ file `IMDB Dataset.csv` để quan sát cấu trúc văn bản phim chân thực.

In [6]:
DATA_PATH = os.path.join("..", "data", "IMDB Dataset.csv")
if os.path.exists(DATA_PATH):
    df_sample = pd.read_csv(DATA_PATH, nrows=3)
    review_sample = df_sample['review'].iloc[0][:200] + "..."
    print("Mẫu review trích từ IMDb:")
    print(review_sample)
    
    enc_review = tokenizer(review_sample, return_tensors="pt")
    rev_tokens = tokenizer.convert_ids_to_tokens(enc_review['input_ids'][0])
    print(f"\nSố lượng token được tạo ra: {len(rev_tokens)}")
    print("20 tokens đầu tiên:", rev_tokens[:20])
else:
    print("Không tìm thấy file dữ liệu IMDb tại", DATA_PATH)

Mẫu review trích từ IMDb:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Số lượng token được tạo ra: 54
20 tokens đầu tiên: ['[CLS]', 'one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'", 'll', 'be', 'hooked']


## 7. Thử nghiệm `max_length`, Padding và Truncation
Trong mini-batch training, các câu có độ dài không đồng đều. Cơ chế của Hugging Face Tokenizer:
- **`truncation=True`**: Cắt ngắn các chuỗi dài hơn `max_length` (giữ lại `[CLS]` ở đầu và chèn `[SEP]` ở cuối).
- **`padding='max_length'`**: Đệm thêm các token `[PAD]` (ID 0) cho tới khi độ dài chuỗi đạt đúng `max_length`.
- **`attention_mask`**: Nhận giá trị `1` cho các token nội dung thực và `0` cho các token `[PAD]`. Khi đưa vào cơ chế Scaled Dot-Product Attention, các vị trí có mask = 0 sẽ bị cộng một lượng âm vô cùng ($-\infty pprox -10^9$) để xác suất Softmax bằng $0$, triệt tiêu ảnh hưởng của padding.

In [7]:
batch_sentences = [
    "Short review.",
    "A moderate length movie review discussing the plot and characters in great detail.",
    "An extremely long review that goes on and on analyzing every aspect of the cinematography, acting, musical score, and visual effects."
]

for max_len in [8, 16, 32]:
    batch_enc = tokenizer(
        batch_sentences,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    print(f"=== Thử nghiệm với max_length = {max_len} ===")
    print("input_ids shape:      ", batch_enc["input_ids"].shape)
    print("attention_mask shape: ", batch_enc["attention_mask"].shape)
    print("Câu 1 input_ids:      ", batch_enc["input_ids"][0].tolist())
    print("Câu 1 attention_mask: ", batch_enc["attention_mask"][0].tolist())
    print("Câu 1 decoded text:   ", tokenizer.decode(batch_enc["input_ids"][0]))
    print()

=== Thử nghiệm với max_length = 8 ===
input_ids shape:       torch.Size([3, 8])
attention_mask shape:  torch.Size([3, 8])
Câu 1 input_ids:       [101, 2460, 3319, 1012, 102, 0, 0, 0]
Câu 1 attention_mask:  [1, 1, 1, 1, 1, 0, 0, 0]
Câu 1 decoded text:    [CLS] short review. [SEP] [PAD] [PAD] [PAD]

=== Thử nghiệm với max_length = 16 ===
input_ids shape:       torch.Size([3, 16])
attention_mask shape:  torch.Size([3, 16])
Câu 1 input_ids:       [101, 2460, 3319, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Câu 1 attention_mask:  [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Câu 1 decoded text:    [CLS] short review. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]

=== Thử nghiệm với max_length = 32 ===
input_ids shape:       torch.Size([3, 32])
attention_mask shape:  torch.Size([3, 32])
Câu 1 input_ids:       [101, 2460, 3319, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Câu 1 attention_mask:  [1, 1, 1, 1, 1, 0, 0, 

## 8. Bảng chi tiết minh họa Attention Mask và Padding
Xem xét trực quan từng vị trí token của một câu ngắn khi bị đệm lên độ dài 10:

In [8]:
short_enc = tokenizer(
    "Good movie!",
    padding="max_length",
    max_length=10,
    return_tensors="pt"
)
tok_list = tokenizer.convert_ids_to_tokens(short_enc["input_ids"][0])
id_list = short_enc["input_ids"][0].tolist()
mask_list = short_enc["attention_mask"][0].tolist()

mask_df = pd.DataFrame({
    "Vị trí (Index)": list(range(len(tok_list))),
    "Token chuỗi": tok_list,
    "Input ID": id_list,
    "Attention Mask": mask_list,
    "Xử lý trong Self-Attention": [
        "Tính toán trọng số Attention bình thường" if m == 1 else "Gán Score = -inf (Xác suất Softmax = 0)"
        for m in mask_list
    ]
})
mask_df

,Vị trí (Index),Token chuỗi,Input ID,Attention Mask,Xử lý trong Self-Attention
0,0,[CLS],101,1,Tính toán trọng số Attention bình thường
1,1,good,2204,1,Tính toán trọng số Attention bình thường
2,2,movie,3185,1,Tính toán trọng số Attention bình thường
3,3,!,999,1,Tính toán trọng số Attention bình thường
4,4,[SEP],102,1,Tính toán trọng số Attention bình thường
5,5,[PAD],0,0,Gán Score = -inf (Xác suất Softmax = 0)
6,6,[PAD],0,0,Gán Score = -inf (Xác suất Softmax = 0)
7,7,[PAD],0,0,Gán Score = -inf (Xác suất Softmax = 0)
8,8,[PAD],0,0,Gán Score = -inf (Xác suất Softmax = 0)
9,9,[PAD],0,0,Gán Score = -inf (Xác suất Softmax = 0)


## 9. Tổng kết các điểm then chốt (Key Takeaways)
1. **Subword Tokenization (WordPiece)** giúp giải quyết hoàn hảo bài toán OOV, giữ cho kích thước từ điển nhỏ gọn (~30k tokens) trong khi vẫn biểu diễn được lượng từ vựng không giới hạn.
2. **Special Tokens (`[CLS]`, `[SEP]`)** đóng vai trò kiến trúc then chốt: `[CLS]` tích lũy biểu diễn ngữ cảnh toàn câu để phân loại, `[SEP]` phân tách các ranh giới.
3. **Attention Mask** là cơ chế phần cứng bắt buộc để huấn luyện batch hiệu quả, ngăn chặn padding tokens làm sai lệch biểu diễn ngữ nghĩa.
4. Việc lựa chọn **`max_length`** là sự đánh đổi trực tiếp giữa độ chi tiết ngữ cảnh và chi phí tính toán $\mathcal{O}(T^2)$ của Transformer Encoder.